## Test Evaluation

The aim of this notebook is to create portfolios for each combination of model and metric on test data and then explore the results.

## Setup

In [1]:
import itertools
import os
from pathlib import Path
from typing import get_args

import pandas as pd
from joblib import Parallel, delayed

In [2]:
# Find project root (folder that contains .git)
ROOT = Path.cwd()
while not (ROOT / ".git").exists():
    ROOT = ROOT.parent

# Set working directory to root
os.chdir(ROOT)

print("Now working in:", Path.cwd())

Now working in: C:\Users\couch\OneDrive\Assignments\Master Thesis\Repo


In [3]:
from src.config import TEST_MODELS
from src.metrics import MetricName

In [4]:
def run_single_combination(params):
    import os
    import sys
    from pathlib import Path

    # Find project root (folder that contains .git)
    ROOT = Path.cwd()
    while not (ROOT / ".git").exists():
        ROOT = ROOT.parent

    # Set working directory AND update Python's import path
    os.chdir(ROOT)
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))

    # Suppress all tqdm progress bars in this process
    os.environ["TQDM_DISABLE"] = "1"

    from src.config import TEST_END, TEST_START, WINDOW_SIZE
    from src.simulation import run_backtest

    metric, model_tuple = params
    mean_model, volatility_model, _ = model_tuple

    results_dir = Path("results/test_evaluation")

    combo_dir = results_dir / metric
    data_dir = combo_dir / "data"
    logs_dir = combo_dir / "logs"

    data_dir.mkdir(parents=True, exist_ok=True)
    logs_dir.mkdir(parents=True, exist_ok=True)

    result_file = data_dir / f"{mean_model}_{volatility_model}.parquet"
    log_file = logs_dir / f"{mean_model}_{volatility_model}.txt"

    if result_file.exists():
        return

    df = pd.read_parquet("results/processed_data/stock_data.parquet", engine="pyarrow")
    df.index = pd.to_datetime(df.index)
    df = df[df.index <= TEST_END]

    run_backtest(
        df,
        window_size=WINDOW_SIZE,
        start_date=TEST_START,
        end_date=TEST_END,
        mean_model=mean_model,
        volatility_model=volatility_model,
        optimize_portfolio_flag=True,
        save_path=result_file,
        ga_metric=metric,
        rf_rates_path="results/processed_data/risk_free_rate.json",
        log_path=log_file,
    )

    print(f"Finished ({mean_model}, {volatility_model}) for {metric}", flush=True)


param_combinations = list(itertools.product(list(get_args(MetricName)), TEST_MODELS))

Parallel(n_jobs=4, backend="loky", batch_size=1)(delayed(run_single_combination)(p) for p in param_combinations)
print("Finished backtest for all combinations")

C:\Users\couch\OneDrive\Assignments\Master Thesis\Repo\.venv\Lib\site-packages\joblib\externals\loky\process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Finished backtest for all combinations
